<a href="https://colab.research.google.com/github/Shalom-u5841413/Shalomu5841413-AnalyticsEngineering/blob/main/2_03_practice_bike_hire_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [2]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [3]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [4]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

In [5]:

# gets the table containing bike journeys
trips = tables["trips"]

# shows the first 5 journeys
display(trips.head())

# counts missing information
print(trips.isna().sum())

# counts completely repeated rows
print(trips.duplicated().sum())

# shows useful stats about journey
print(trips["duration_min"].describe())


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time           0
duration_min        30
user_type            0
dtype: int64
12
count     582.000000
mean       27.891753
std       145.064305
min         4.000000
25%        12.000000
50%        16.000000
75%        22.000000
max      2880.000000
Name: duration_min, dtype: float64


i'm now going to investigate the repeated journeys discovered in the above code. i expect the code to show the repeated entries with duplicated values

In [6]:

display(trips[trips.duplicated(keep=False)].head(10))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
96,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
97,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
175,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
176,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
185,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
186,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
285,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
286,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member


after running the code i have discovered that there are multiple entries that are repeated e.g. two entries havr the same trip ID of T0323 and where taken on the exact same day and time.

this can negatively impact subsequent calculations by giving some journeys more weight than others



```

display(trips[trips.duplicated(keep=False)].head(10))

```



## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

now i'm going to remove the extra copies and i expect the number of rows to decrease by 12

In [7]:

# Make a copy so the original data is unchanged
trips_clean = trips.copy()

# Remove exact duplicate rows
trips_clean = trips_clean.drop_duplicates()

# Compare the number of rows
print("Before:", len(trips))
print("After:", len(trips_clean))

# Check whether duplicates remain
print("Duplicates remaining:", trips_clean.duplicated().sum())


Before: 612
After: 600
Duplicates remaining: 0


the number of rows decreased from 612 to 600 as expected. and duplicates remaining shows 0 and ensures that only the repeat entries are removed.

this is useful to do before calculating journey durations in the next code chunk as it could distort the results

now i'm going to check how many missing journey durations are in the the cleaned data set "trips_clean". this is to understand how much information is unavailable before calculating averages.

OR

this is because the manager wants to know if electric bikes are used more than standard ones and we can't calculate that with missing values

In [8]:

print("Missing durations:", trips_clean["duration_min"].isna().sum())

display(trips_clean[trips_clean["duration_min"].isna()].head())


Missing durations: 30


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
12,T0013,B024,S04,S02,01/03/2025 18:43,NaN,Member
14,T0015,B040,S03,S04,2025-03-01 19:03:00,NaN,Member
30,T0031,B015,S06,S06,2025-03-02 08:50:00,NaN,Casual
60,T0061,B011,S09,S07,2025-03-04 08:23:00,NaN,Member
64,T0065,B041,S06,S02,04/03/2025 09:06,NaN,Member


there's still 30 missing durations in the cleaned data. i've decided to leave them in the data set and use python's .mean() to automatically exclude them from the mean calculations as they might still retain relevant information for the analysis.

also i've noticed the date format varies. potentially another data- cleaning issue i'll investigate later

also, i noticed in earlier output that the longest journey was 2,880 minutes while the median was 16 minutes. i think this is a disparity worth investigating as it might have distorted average journey calculation.

i'm going to check this by counting how many journeys are longer than 600 minutes (10 hours), then depending on that frequency, i'll know how common unusually long journeys are.

**AI prompt:** Using Python Pandas, show me how to count and display journeys in the DataFrame trips_clean where duration_min is greater than 600. Do not remove or change any data. Explain what each line of code does.

In [9]:

# Display journeys lasting longer than 600 minutes
display(trips_clean[trips_clean["duration_min"] > 600])

# Count how many journeys exceed 600 minutes
print("Journeys over 600 minutes:",
      (trips_clean["duration_min"] > 600).sum())


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
80,T0081,B010,S12,S11,2025-03-04 21:57:00,2880.0,Member
143,T0143,B020,S09,S10,2025-03-08 12:23:00,1440.0,Casual
287,T0284,B046,S08,S03,2025-03-15 20:25:00,1440.0,Member


Journeys over 600 minutes: 3


i encountered a NameError when running the above code chunk because i was disconnected from the collab session. i reran earlier set ups to recreate the table and the code ran successfully

after running the code, it showed that 3 journeys exceeded 600 minutes. 2 of the records had the same time of 1440 minutes. i don't know how to check if they're geniune journeys and excluding them from the data will reduce the average. i'll keep the data but exclude the journeys >600 from the calculations

### AI prompts and decisions
**AI prompt:**

**AI suggestion:**

**My decison and check:**

now i want to create a new duration column that excludes missing values and journeys longer than 600 minutes from the average without deleting their records.

i beleive the duration column will reduce by 33 (30 missing durations + 3 >600 durations), therefore reuding the overall average

**AI Prompt:** i've found 3 journeys in trips_clean that are over 600 minutes. i want to exclude these from the average calculation but keep the rows in the data as they might still have useful information. can you give me python code to do this and compare the average before and after? also check that the number of rows hasn't changed.

In [10]:
import numpy as np

# Create a new column with durations > 600 masked as NaN
trips_clean['duration_filtered'] = np.where(trips_clean['duration_min'] > 600, np.nan, trips_clean['duration_min'])

# Calculate averages
avg_before = trips_clean['duration_min'].mean()
avg_after = trips_clean['duration_filtered'].mean()

# Verify rows and display results
print(f"Row count before: {len(trips_clean)}")
print(f"Row count after: {len(trips_clean)}")
print(f"Average duration before (including >600): {avg_before:.2f} minutes")
print(f"Average duration after (excluding >600): {avg_after:.2f} minutes")

# Display the rows that were masked for verification
display(trips_clean[trips_clean['duration_min'] > 600][['trip_id', 'duration_min', 'duration_filtered']])

Row count before: 600
Row count after: 600
Average duration before (including >600): 28.05 minutes
Average duration after (excluding >600): 18.04 minutes


,trip_id,duration_min,duration_filtered
80,T0081,2880.0,NaN
143,T0143,1440.0,NaN
287,T0284,1440.0,NaN


gemini has correctly created a separate column for the average calculation, but it's before and after row count check is repetitve and doesn't generate the output i'm looking for. the code below is a more accurate depiction.

In [11]:
print("Rows:", len(trips_clean))
print("Original durations still present:",
      trips_clean["duration_min"].notna().sum())
print("Filtered durations present:",
      trips_clean["duration_filtered"].notna().sum())

Rows: 600
Original durations still present: 570
Filtered durations present: 567


as predicted the duration column dropped by 33 and the average dropped as well. turning the >600 journeys to NAs efficiently excluded them from the calculations when using python's .mean(function)

next i'm checking whether elictrical and standard bikes are labelled consistently because inconsistent labels could affect the comparison between bike types. i expect this to count how many bikes belong to each recorded category, including missing values.

**AI Prompt:** i want to check the different bike types in the bikes table to see if there are any inconsistencies in how they're recorded. can you give me python code to show the different values and how often they appear? don't change anything yet.

In [12]:
# Access the bikes table
bikes = tables["bikes"]

# Count the occurrences of each unique bike type
bike_type_counts = bikes["bike_type"].value_counts(dropna=False)

# Display the unique types and counts
print("Unique bike types and counts:")
display(bike_type_counts)

Unique bike types and counts:


,count
bike_type,
Standard,36
Electric,9
STANDARD,8
standard,3
Electric,2
electric,1
E-bike,1


due to inconsistent formatting, the code counts things like "STANDARD" and "Standard" as two different types.

so, i'm going to fix this by standardising the bike labels to avoid treating the same bike type as different categories.

**AI Prompt:** i've noticed that the bike types are recorded differently even though some mean the same thing. can you give me code to make them consistent in a new bikes_clean table, without changing the original? also show me the values before and after so i can check it worked.

In [13]:
# Create a clean copy of the bikes dataframe
bikes_clean = tables["bikes"].copy()

# Standardize bike_type using string operations and a mapping dictionary
# First, standardize casing by turning everything lowercase and stripping whitespace
bikes_clean['bike_type'] = bikes_clean['bike_type'].str.strip().str.lower()

# Map the standardized lowercase values to clean categories
mapping = {
    'standard': 'Standard',
    'electric': 'Electric',
    'e-bike': 'Electric'
}
bikes_clean['bike_type'] = bikes_clean['bike_type'].map(mapping)

# Display "Before" and "After" categories to verify
print("=== BEFORE CLEANING ===")
print(tables["bikes"]["bike_type"].value_counts(dropna=False))

print("\n=== AFTER CLEANING ===")
print(bikes_clean["bike_type"].value_counts(dropna=False))

=== BEFORE CLEANING ===
bike_type
Standard     36
Electric      9
STANDARD      8
standard      3
Electric      2
electric      1
E-bike        1
Name: count, dtype: int64

=== AFTER CLEANING ===
bike_type
Standard    47
Electric    13
Name: count, dtype: int64


the seven labels were combined into two categories, with all 60 records preserved.

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

now i'm going to combine the 3 separate tables so that each journey has its bike type and starting zone, i expect the 600 journeys to remain..

**AI Prompt:** i've now cleaned the trips and bikes tables. i want to combine trips_clean with bikes_clean using bike_id, then add the zone from the stations table using start_station_id. can you give me python code to do this and check that the number of journeys hasn't increased? i want to keep all 600 journeys.

In [14]:
# 1. Merge trips_clean with bikes_clean using bike_id
merged_trips_bikes = trips_clean.merge(bikes_clean, on='bike_id', how='left')

# 2. Merge with stations on start_station_id to add the station zone
# Access stations from the tables dictionary
stations = tables["stations"]

final_merged = merged_trips_bikes.merge(
    stations[['station_id', 'zone']],
    left_on='start_station_id',
    right_on='station_id',
    how='left'
)

# 3. Print the row count checks to verify no journeys were added or lost
print(f"Trips in trips_clean: {len(trips_clean)}")
print(f"Trips after merging with bikes: {len(merged_trips_bikes)}")
print(f"Trips after merging with stations: {len(final_merged)}")

# Quick confirmation check
if len(final_merged) == 600:
    print("Success! The number of journeys is exactly 600.")
else:
    print(f"Warning: The journey count has changed to {len(final_merged)}.")

# Preview the final joined DataFrame
display(final_merged.head())

Trips in trips_clean: 600
Trips after merging with bikes: 600
Trips after merging with stations: 600
Success! The number of journeys is exactly 600.


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type,duration_filtered,bike_type,purchase_year,station_id,zone
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual,21.0,Standard,2021,S05,South
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL,27.0,Standard,2021,S12,Centre
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual,18.0,Standard,2024,S11,West
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member,34.0,Electric,2024,S04,West
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member,16.0,Standard,2023,S01,Centre


that came out as expected but i still need to check whether any bike types or zones are missing. keeping 600 rows doesn't automatically mean every journey matched successfully.

In [15]:

print("Missing bike types:", final_merged["bike_type"].isna().sum())
print("Missing zones:", final_merged["zone"].isna().sum())


Missing bike types: 0
Missing zones: 0


both results being zero, makes me more confident that the information has combined correctly.

now that i've combined the tables, i want to calculate the average journey duration for electric and standard bikes in each starting zone. i'll use duration_filtered so the missing durations and journeys over 600 minutes aren't included in the averages. i'll also count the journeys in each group to check the results.

**AI Prompt:** i've combined my tables into final_merged and want to compare the average journey duration for electric and standard bikes in each starting zone. can you give me python code to calculate this using duration_filtered? also show how many journeys were used for each average so i can check the results.

In [16]:
# Calculate the average duration and the count of journeys grouped by bike type and starting zone
final_summary = final_merged.groupby(['bike_type', 'zone']).agg(
    number_of_trips=('trip_id', 'count'),
    avg_duration_minutes=('duration_filtered', 'mean')
).reset_index()

# Round the average duration to 2 decimal places for cleaner reporting
final_summary['avg_duration_minutes'] = final_summary['avg_duration_minutes'].round(2)

# Display the final summary
display(final_summary)

,bike_type,zone,number_of_trips,avg_duration_minutes
0,Electric,Centre,87,14.51
1,Electric,East,12,15.73
2,Electric,North,24,15.96
3,Electric,South,26,13.42
4,Electric,West,25,15.68
5,Standard,Centre,227,18.93
6,Standard,East,24,19.50
7,Standard,North,39,20.36
8,Standard,South,60,19.47
9,Standard,West,76,19.82


the results show that standard bikes have longer average journeys than electric bikes across all 5 zones. for example, in the Centre, standard bikes averaged 18.93 minutes compared to 14.51 minutes for electric bikes.

however, i've noticed that gemini counted all the trips rather than just the ones used to calculate the averages. this means the number_of_trips column might include journeys with missing or excluded durations, so i'll correct this before drawing my final conclusion.

In [17]:
# Calculate the average duration and the exact count of journeys with valid durations (excluding nulls and >600 mins)
final_summary_fixed = final_merged.groupby(['bike_type', 'zone']).agg(
    total_journeys_recorded=('trip_id', 'count'),
    journeys_used_for_average=('duration_filtered', 'count'),
    avg_duration_minutes=('duration_filtered', 'mean')
).reset_index()

# Round the average duration to 2 decimal places
final_summary_fixed['avg_duration_minutes'] = final_summary_fixed['avg_duration_minutes'].round(2)

# Display the corrected final summary
display(final_summary_fixed)

,bike_type,zone,total_journeys_recorded,journeys_used_for_average,avg_duration_minutes
0,Electric,Centre,87,81,14.51
1,Electric,East,12,11,15.73
2,Electric,North,24,23,15.96
3,Electric,South,26,24,13.42
4,Electric,West,25,22,15.68
5,Standard,Centre,227,215,18.93
6,Standard,East,24,20,19.50
7,Standard,North,39,39,20.36
8,Standard,South,60,59,19.47
9,Standard,West,76,73,19.82


after correcting the code, the number of journeys used for the averages is now 567 rather than 600. this makes sense because 30 durations were originally missing and another 3 were excluded for being over 600 minutes.

the averages have stayed the same, so the issue was with how gemini counted the journeys rather than the average calculations themselves.

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

## 6. What I could not verify

*What remains unchecked? What would you do next?*